# 04 - WOWA Scraping

This notebook scrapes the Canadian REIT competitor table from WOWA.

Final CSV output:
- `data/wowa_competitors.csv`

In [1]:
!pip -q install requests beautifulsoup4 lxml pandas

In [2]:
import re
from pathlib import Path
import pandas as pd
import requests
from bs4 import BeautifulSoup
try:
    from IPython.display import display
except ImportError:
    # Local validation fallback. Colab has display(), but plain Python may not.
    def display(value):
        print(value)


DATA_DIR = Path("/content/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
WOWA_URL = "https://wowa.ca/reit-canada"
COMPETITOR_KEYWORDS = ["SmartCentres", "RioCan", "Choice", "Allied", "First Capital"]
session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0 (MBA assignment basic scraper)"})

# Local Windows certificate stores can fail for public HTTPS requests. Colab
# usually works normally, but this keeps the notebook runnable on this PC too.
session.verify = False
requests.packages.urllib3.disable_warnings()

In [3]:
def clean_number(value):
    match = re.search(r"-?\d+(\.\d+)?", str(value or "").replace(",", ""))
    return float(match.group(0)) if match else None

def parse_html_table(table):
    headers = [cell.get_text(" ", strip=True) for cell in table.select("thead th")]
    if not headers:
        first_row = table.find("tr")
        headers = [cell.get_text(" ", strip=True) for cell in first_row.find_all(["th", "td"])] if first_row else []
    rows = []
    for tr in table.select("tbody tr"):
        cells = [cell.get_text(" ", strip=True) for cell in tr.find_all(["td", "th"])]
        if len(cells) == len(headers):
            rows.append(dict(zip(headers, cells)))
    return rows

In [4]:
response = session.get(WOWA_URL, timeout=30)
response.raise_for_status()
soup = BeautifulSoup(response.text, "html.parser")

# The useful table has REIT and Market Cap columns.
all_rows = []
for table in soup.find_all("table"):
    rows = parse_html_table(table)
    if not rows:
        continue
    header_text = " ".join(rows[0].keys()).lower()
    if "reit" in header_text and "market cap" in header_text:
        all_rows = rows
        break

filtered_rows = []
for row in all_rows:
    row_text = " ".join(str(value) for value in row.values())
    if any(keyword.lower() in row_text.lower() for keyword in COMPETITOR_KEYWORDS):
        cleaned = dict(row)
        cleaned["source"] = "WOWA"
        cleaned["share_price_numeric"] = clean_number(row.get("Share Price"))
        cleaned["market_cap_numeric"] = clean_number(row.get("Market Cap"))
        cleaned["yield_numeric"] = clean_number(row.get("Yield"))
        filtered_rows.append(cleaned)

wowa_competitors_df = pd.DataFrame(filtered_rows)
wowa_competitors_df.to_csv(DATA_DIR / "wowa_competitors.csv", index=False)
print("Saved competitors:", len(wowa_competitors_df))
display(wowa_competitors_df)

Saved competitors: 5


,REIT,Symbol (TSX),Type,Share Price,Market Cap,Yield,source,share_price_numeric,market_cap_numeric,yield_numeric
0,RioCan REIT,REI.UN,Retail,$21.90,$6.93 Billion,4.39%,WOWA,21.90,6.93,4.39
1,Allied Properties REIT,AP.UN,Office,$42.61,$5.45 Billion,3.97%,WOWA,42.61,5.45,3.97
2,Choice Properties REIT,CHP.UN,Commercial / Residential,$15.07,$4.87 Billion,4.97%,WOWA,15.07,4.87,4.97
3,SmartCentres REIT,SRU.UN,Retail,$30.00,$4.35 Billion,6.15%,WOWA,30.00,4.35,6.15
4,First Capital Realty,FCR.UN,Commercial,$17.65,$3.84 Billion,2.47%,WOWA,17.65,3.84,2.47


Source note: WOWA and yfinance may show different yields or market caps because they are different sources and may update at different times. Keep the source column when comparing them later.